In [ ]:
# ═══════════════════════════════════════════════════════════════
# ENDING VOLUME — PENETRATION STUDY (brake calibration)
# ═══════════════════════════════════════════════════════════════
#
# Purpose:   Answer ONE question before any backtest runs:
#            how far does price poke through the volume candle's
#            low (longs) / high (shorts) before it either works
#            or genuinely fails?
#
#            That number sets the HARD BRAKE, which sets R,
#            which sets position size and the case 1 / case 2
#            branch. Everything downstream depends on it, so it
#            gets measured, not guessed.
#
# ─────────────────────────────────────────────────────────────
# WHY THIS EXISTS
# ─────────────────────────────────────────────────────────────
#
#   $MSFU  → highest volume of session at $39, next candle
#            breaks slightly higher, kills the short. But it
#            was pennies. We shouldn't have been out.
#
#   $NVDL  → ending volume 10:00A, 10:05A prints EVEN HIGHER
#            volume and nibbles the low. Stopped, then it
#            bounced to the moon without us.
#
#   A stop exactly at the candle's extreme gets picked off by
#   noise. A stop too far away stops being a stop. This measures
#   where the line actually sits.
#
# ─────────────────────────────────────────────────────────────
# WHAT IT MEASURES
# ─────────────────────────────────────────────────────────────
#
#   For every qualifying volume candle:
#
#     penetration  = how far past the candle's extreme price
#                    traded afterward
#     normalized   = penetration ÷ that candle's own range
#                    (a 2c poke on a 15c candle is a very
#                     different animal than 2c on a 60c candle)
#     outcome      = did price reach VWAP afterward, or not
#
#   Then: among the ones that WORKED, how deep did the poke
#   get? That percentile is the brake.
#
#   NOTE: this measures raw price behavior only. No entries,
#   no exits, no stop rules — so it can't quietly assume the
#   answer the way a P&L sweep can.
#
# ─────────────────────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────────────────────
#
#   ENDVOL_penetration_raw.csv       one row per signal candle
#   ENDVOL_penetration_summary.csv   percentiles + brake rec
#
# ═══════════════════════════════════════════════════════════════

%pip install alpaca-py
%pip install pandas python-dotenv pyarrow

In [ ]:
# ─── Cell 2: Imports ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import datetime as dt
from datetime import datetime
from dotenv import load_dotenv

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

print("✅ Libraries imported")

In [ ]:
# ─── Cell 3: Alpaca Connection ────────────────────────────────
BASE_PATH = (
    r"c:\Users\kwgit\Desktop\Playbook"
    r"\Claude_2025_2026_Playbooks"
    r"\AlgoTrading\AlgoTrading"
)

dotenv_path = BASE_PATH + r"\.env"
load_dotenv(dotenv_path, override=True)

API_KEY    = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

# Data client — live keys for market data
client = StockHistoricalDataClient(
    api_key    = API_KEY,
    secret_key = SECRET_KEY
)

print("✅ Alpaca connected")
print(f"   Data client:     ✅ Live keys")

In [ ]:
# ─── Cell 4: Configuration ────────────────────────────────────

WATCHLIST_FILE = BASE_PATH + r"\ENDVOL_watchlist.txt"
CACHE_DIR      = BASE_PATH + r"\ENDVOL_cache"
RAW_OUT        = BASE_PATH + r"\ENDVOL_penetration_raw.csv"
SUMMARY_OUT    = BASE_PATH + r"\ENDVOL_penetration_summary.csv"

# ── Date range ────────────────────────────────────────────────
START_DATE = "2023-01-01"
END_DATE   = "2026-08-31"

# ── Feed ──────────────────────────────────────────────────────
# SIP = full consolidated tape. The free plan allows historical
# SIP as long as the query end is >15 min old — ours is years.
# IEX would be ~2.5% of volume, which would make every
# "highest volume candle" ranking meaningless.
FEED = DataFeed.SIP

# ── Session windows (Eastern) ─────────────────────────────────
SESSION_OPEN  = dt.time(9, 30)
SESSION_CLOSE = dt.time(16, 0)
OR_END        = dt.time(9, 45)   # 15-min opening range ends

# Rank path can only FIRE after the opening range.
# On 2-min the 9:44 bar spans 9:44-9:46, so the first bar
# fully clear of the range starts at 9:46.
RANK_START_2M = dt.time(9, 46)
RANK_START_5M = dt.time(9, 45)

# 2x path can fire earlier — it just needs one prior candle
# to measure against.
X2_START_2M   = dt.time(9, 32)
X2_START_5M   = dt.time(9, 35)

# ── Signal criteria ───────────────────────────────────────────
TOP_RANK      = 2      # new #1 or #2 volume candle of session
X2_MULTIPLE   = 2.0    # or >= 2x the session's reigning biggest

# ── Study parameters ──────────────────────────────────────────
PRICE_FLOOR   = 4.00   # same floor as the discovery scanner
MIN_BAR_VOL   = 1_000  # ignore candles too thin to trade

# ── Data quality guards ───────────────────────────────────────
# Alpaca returns bars only where trades actually printed. A
# young or thin ticker can have NO bars before 9:45 — and then
# the "opening range" gets built from one stray candle, which
# makes every later bar look like a breakout and assigns
# direction off noise. Guard against it.
#
# Full opening range = 8 bars on 2-min (9:30..9:44),
#                      3 bars on 5-min (9:30, 9:35, 9:40)
MIN_OR_BARS_2M   = 6      # of 8
MIN_OR_BARS_5M   = 3      # of 3 — need all of them
MIN_SESSION_FRAC = 0.80   # session must have >= 80% of its
                          # expected bars to be trustworthy
FULL_BARS = {"2m": 195, "5m": 78}

# ── Segmentation ──────────────────────────────────────────────
# When the watchlist is widened to the full S&P/NDX, the index
# names will swamp the sample. Tag every row so the analysis can
# separate "names Kevin actually trades" from the broad universe
# instead of pooling them into one misleading average.
PLAYBOOK_NAMES = {
    'SNAP','MRVL','DKNG','PLTU','SMCX','NVDL','SNXX','AMZU','MSFU',
    'AAPU','TSLL','SPCH','SSPC','MRNA','SKHX','MVLL','IBIT','CONL',
    'AMDL','SNDQ','CMG','CRMG','AFRM','GME','DLLL','PL','IOT','HPE','MUU',
}

print("✅ Configuration loaded")
print(f"   Range:        {START_DATE} → {END_DATE}")
print(f"   Feed:         {FEED.value.upper()}")
print(f"   Rank path:    new #1 or #{TOP_RANK} of session")
print(f"   2x path:      >= {X2_MULTIPLE}x reigning biggest candle")

In [ ]:
# ─── Cell 5: Universe Loader ──────────────────────────────────

def read_ticker_file(path, label):
    """Read one ticker file, skip # comments and blanks"""
    if not os.path.exists(path):
        print(f"   ⚠️  {label}: not found at {path}")
        return []
    with open(path, 'r') as f:
        tickers = [
            line.strip().upper()
            for line in f
            if line.strip() and not line.strip().startswith('#')
        ]
    print(f"   {label:<10} {len(tickers):>4} tickers")
    return tickers


universe = read_ticker_file(WATCHLIST_FILE, "ENDVOL")
universe = sorted(set(t for t in universe if '.' not in t))

os.makedirs(CACHE_DIR, exist_ok=True)

print(f"\n   ✅ Universe: {len(universe)} unique tickers")
print(f"   ✅ Cache dir: {CACHE_DIR}")

In [ ]:
# ─── Cell 6: Bar Fetch + Cache ────────────────────────────────
#
# 3.7 years of 2-min bars across the watchlist is a lot of
# pulling. Cache each ticker/timeframe to parquet so re-runs
# and re-tuning cost nothing.

TF = {
    "2m": TimeFrame(2, TimeFrameUnit.Minute),
    "5m": TimeFrame(5, TimeFrameUnit.Minute),
}


def cache_path(symbol, tf_label):
    return os.path.join(CACHE_DIR, f"{symbol}_{tf_label}.parquet")


def fetch_bars(symbol, tf_label, force=False):
    """
    Pull regular-hours bars for one ticker/timeframe over the
    full study range. Returns tz-aware (America/New_York) frame
    indexed by bar start time.
    """
    path = cache_path(symbol, tf_label)
    if os.path.exists(path) and not force:
        return pd.read_parquet(path)

    req = StockBarsRequest(
        symbol_or_symbols = symbol,
        timeframe         = TF[tf_label],
        start             = START_DATE,
        end               = END_DATE,
        feed              = FEED,
    )
    df = client.get_stock_bars(req).df

    if df is None or len(df) == 0:
        empty = pd.DataFrame(
            columns=['open','high','low','close','volume'])
        empty.to_parquet(path)
        return empty

    if isinstance(df.index, pd.MultiIndex):
        df = df.xs(symbol, level=0)

    df = df[['open','high','low','close','volume']].copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df.index = df.index.tz_convert('America/New_York')

    # Regular hours only — 9:30 to 4:00. Pre/post volume would
    # pollute every "highest of session" ranking.
    df = df[(df.index.time >= SESSION_OPEN) &
            (df.index.time <  SESSION_CLOSE)]

    df.to_parquet(path)
    return df


# ── Smoke test before committing to the full pull ────────────
print("Smoke test — NVDL 2m, verifying SIP access...\n")
_test = fetch_bars("NVDL", "2m", force=True)

if len(_test) == 0:
    print("   ❌ No bars returned. Check symbol / date range / feed.")
else:
    _days = _test.index.normalize().nunique()
    print(f"   ✅ {len(_test):,} bars across {_days} sessions")
    print(f"   ✅ First: {_test.index.min()}")
    print(f"   ✅ Last:  {_test.index.max()}")

    # ── Coverage check ────────────────────────────────────────
    # Alpaca emits bars only where trades printed, so a thin
    # name has holes. Sessions missing their opening range
    # can't be studied — the range would be fiction.
    _per = _test.groupby(_test.index.date).size()
    _orb = (_test[_test.index.time < OR_END]
            .groupby(_test[_test.index.time < OR_END].index.date).size())
    _orb = _orb.reindex(_per.index, fill_value=0)

    print(f"\n   Bars per session:  median {_per.median():.0f} "
          f"of {FULL_BARS['2m']} possible")
    print(f"   Opening-range bars: median {_orb.median():.0f} of 8")
    print(f"   Sessions with >= {MIN_OR_BARS_2M} OR bars: "
          f"{(_orb >= MIN_OR_BARS_2M).sum()} / {len(_per)} "
          f"({(_orb >= MIN_OR_BARS_2M).mean()*100:.0f}%)")
    print(f"   Sessions >= {MIN_SESSION_FRAC:.0%} full: "
          f"{(_per >= FULL_BARS['2m']*MIN_SESSION_FRAC).sum()} / {len(_per)}")

    _by_year = _orb.groupby([d.year for d in _orb.index]).apply(
        lambda g: (g >= MIN_OR_BARS_2M).mean())
    print(f"\n   Usable-session rate by year:")
    for yr, frac in _by_year.items():
        print(f"      {yr}   {frac*100:5.0f}%")
    print(f"\n   Thin early years get dropped by the guard, which")
    print(f"   is correct — you don't trade an ETF that barely")
    print(f"   prints in its first months.")

In [ ]:
# ─── Cell 7: Signal Detection ─────────────────────────────────
#
# Faithful to the spec:
#
#   Pool         all session bars from 9:30 (opening range
#                candles DO count toward rankings)
#   Rank path    bar becomes the new #1 or #2 by volume;
#                may only FIRE after the opening range
#   2x path      bar >= 2x the session's reigning biggest
#                candle; may fire from 9:32 (2m) / 9:35 (5m)
#   Direction    counter-trend off the 15-min opening range.
#                Spike below OR low  → LONG
#                Spike above OR high → SHORT
#                Pre-9:45 (2x only, no range yet) → low of day
#                is LONG, high of day is SHORT, else vs open


def session_vwap(day_df):
    """Session VWAP from 9:30, typical-price weighted"""
    tp = (day_df['high'] + day_df['low'] + day_df['close']) / 3.0
    cum_pv  = (tp * day_df['volume']).cumsum()
    cum_vol = day_df['volume'].cumsum()
    return cum_pv / cum_vol.replace(0, np.nan)


def find_signals(day_df, tf_label, vwap):
    """
    Return (signals, skip_reason) for one ticker-session.
    skip_reason is None when the session was actually studied.
    """
    min_or = MIN_OR_BARS_2M if tf_label == "2m" else MIN_OR_BARS_5M

    if len(day_df) < FULL_BARS[tf_label] * MIN_SESSION_FRAC:
        return [], 'sparse_session'

    rank_start = RANK_START_2M if tf_label == "2m" else RANK_START_5M
    x2_start   = X2_START_2M   if tf_label == "2m" else X2_START_5M

    # Opening range: 9:30 through 9:45 — must be well covered,
    # or the range is fiction and every breakout call is noise
    or_mask = day_df.index.time < OR_END
    if or_mask.sum() < min_or:
        return [], 'thin_opening_range'

    or_high = day_df.loc[or_mask, 'high'].max()
    or_low  = day_df.loc[or_mask, 'low'].min()
    if not (or_high > or_low):
        return [], 'degenerate_range'

    sess_open = day_df['open'].iloc[0]

    vols  = day_df['volume'].to_numpy()
    highs = day_df['high'].to_numpy()
    lows  = day_df['low'].to_numpy()
    times = day_df.index

    signals = []

    for i in range(1, len(day_df)):
        v  = vols[i]
        t  = times[i].time()
        if v < MIN_BAR_VOL:
            continue

        prior_max = vols[:i].max()
        rank      = int((vols[:i+1] > v).sum()) + 1

        is_x2   = (t >= x2_start)   and prior_max > 0 and v >= X2_MULTIPLE * prior_max
        is_rank = (t >= rank_start) and rank <= TOP_RANK

        if not (is_x2 or is_rank):
            continue

        # ── Direction ─────────────────────────────────────────
        after_or = t >= OR_END
        if after_or:
            below = lows[i]  < or_low
            above = highs[i] > or_high
            if below and above:
                # bar engulfs the whole range — take the further side
                direction = ('long' if (or_low - lows[i]) >= (highs[i] - or_high)
                             else 'short')
            elif below:
                direction = 'long'
            elif above:
                direction = 'short'
            else:
                continue   # inside the range — no trade
        else:
            # Pre-9:45: no completed range yet (2x path only)
            if lows[i] <= lows[:i+1].min():
                direction = 'long'
            elif highs[i] >= highs[:i+1].max():
                direction = 'short'
            else:
                direction = ('long' if day_df['close'].iloc[i] < sess_open
                             else 'short')

        signals.append({
            'bar_idx'    : i,
            'time'       : times[i],
            'tf'         : tf_label,
            'direction'  : direction,
            'path'       : '2x' if is_x2 else 'rank',
            'rank'       : rank,
            'x2_ratio'   : v / prior_max if prior_max > 0 else np.nan,
            'volume'     : v,
            'open'       : day_df['open'].iloc[i],
            'high'       : highs[i],
            'low'        : lows[i],
            'close'      : day_df['close'].iloc[i],
            'candle_rng' : highs[i] - lows[i],
            'vwap_at_sig': vwap.iloc[i],
            'or_high'    : or_high,
            'or_low'     : or_low,
            'pre_or'     : not after_or,
        })

    return signals, None

In [ ]:
# ─── Cell 8: Penetration Measurement ──────────────────────────
#
# For each signal, walk forward and record:
#
#   pen_before_vwap   deepest poke past the candle extreme
#                     BEFORE price first tags VWAP
#   reached_vwap      did it ever tag VWAP that session
#   pen_session       deepest poke at any point through 4:00
#   gapped            did the very next bar OPEN already past
#                     the extreme (a brake couldn't have
#                     filled at its stated price)
#
# The recovered cases are the ones that matter: their
# pen_before_vwap distribution IS the brake question.


def measure_penetration(day_df, sig, vwap_full):
    i     = sig['bar_idx']
    rng   = sig['candle_rng']
    if rng <= 0 or i + 1 >= len(day_df):
        return None

    fwd   = day_df.iloc[i+1:]
    vwap  = vwap_full.iloc[i+1:]
    long_ = sig['direction'] == 'long'

    extreme = sig['low'] if long_ else sig['high']

    if long_:
        pen_series = (extreme - fwd['low']).clip(lower=0)
        hit_vwap   = fwd['high'] >= vwap
        gapped     = fwd['open'].iloc[0] < extreme
    else:
        pen_series = (fwd['high'] - extreme).clip(lower=0)
        hit_vwap   = fwd['low'] <= vwap
        gapped     = fwd['open'].iloc[0] > extreme

    reached = bool(hit_vwap.any())
    if reached:
        first = int(np.argmax(hit_vwap.to_numpy()))
        pen_before = float(pen_series.iloc[:first+1].max())
        bars_to_vwap = first + 1
        vwap_at_hit = float(vwap.iloc[first])
    else:
        pen_before = np.nan
        bars_to_vwap = np.nan
        vwap_at_hit = np.nan

    return {
        'pen_before_vwap'      : pen_before,
        'pen_before_vwap_norm' : pen_before / rng if reached else np.nan,
        'pen_session'          : float(pen_series.max()),
        'pen_session_norm'     : float(pen_series.max()) / rng,
        'reached_vwap'         : reached,
        'bars_to_vwap'         : bars_to_vwap,
        'vwap_at_hit'          : vwap_at_hit,
        'gapped'               : bool(gapped),
    }


def study_ticker(symbol):
    """
    Run detection + measurement across both timeframes.
    Returns (rows, skips) where skips counts why sessions
    were dropped — we want that visible, not silent.
    """
    rows, skips = [], {}

    def note(r):
        skips[r] = skips.get(r, 0) + 1

    for tf_label in ("2m", "5m"):
        try:
            bars = fetch_bars(symbol, tf_label)
        except Exception as e:
            print(f"   ⚠️  {symbol} {tf_label}: "
                  f"{type(e).__name__} {str(e)[:80]}")
            note('fetch_error')
            continue
        if len(bars) == 0:
            note('no_bars')
            continue

        for day, day_df in bars.groupby(bars.index.date):
            if day_df['close'].iloc[0] < PRICE_FLOOR:
                note('below_price_floor')
                continue

            vwap = session_vwap(day_df)
            sigs, reason = find_signals(day_df, tf_label, vwap)
            if reason:
                note(reason)
                continue

            note('studied')
            for sig in sigs:
                m = measure_penetration(day_df, sig, vwap)
                if m is None:
                    continue
                row = {'symbol': symbol, 'session': day,
                       'in_playbook': symbol in PLAYBOOK_NAMES}
                row.update({k: v for k, v in sig.items()
                            if k != 'bar_idx'})
                row.update(m)
                rows.append(row)

    return rows, skips


print("✅ Detection + measurement defined")

In [ ]:
# ─── Cell 9: Run the Study ────────────────────────────────────

# Rows are appended to disk per ticker rather than accumulated in
# memory. At 500+ tickers this is ~2M signals; holding them as a
# list of dicts would run to several GB and could die at hour four
# of a five-hour run.

all_skips  = {}
total_rows = 0
wrote_hdr  = False
t0 = datetime.now()

if os.path.exists(RAW_OUT):
    os.remove(RAW_OUT)

for n, sym in enumerate(universe, 1):
    rows, skips = study_ticker(sym)

    if rows:
        pd.DataFrame(rows).to_csv(
            RAW_OUT, mode='a', header=not wrote_hdr, index=False)
        wrote_hdr = True
    total_rows += len(rows)

    for k, v in skips.items():
        all_skips[k] = all_skips.get(k, 0) + v

    el  = (datetime.now() - t0).total_seconds()
    eta = el / n * (len(universe) - n) / 60
    print(f"   [{n:>3}/{len(universe)}] {sym:<6} "
          f"{len(rows):>6} signals   "
          f"{skips.get('studied', 0):>4} sessions   "
          f"(total {total_rows:,})   ETA {eta:>5.0f} min")

elapsed = (datetime.now() - t0).total_seconds()
print(f"\n✅ Study complete in {elapsed/60:.1f} min")
print(f"   Signals found: {total_rows:,}")
print(f"   Saved → {RAW_OUT}")

print(f"\n   Session accounting (both timeframes):")
for k in sorted(all_skips, key=lambda x: -all_skips[x]):
    label = 'STUDIED' if k == 'studied' else f'dropped: {k}'
    print(f"      {label:<28} {all_skips[k]:>7,}")

pen = pd.read_csv(RAW_OUT) if total_rows else pd.DataFrame()

In [ ]:
# ─── Cell 10: The Answer — Where Does the Brake Go? ───────────

if len(pen) == 0:
    print("No signals. Check the smoke test in Cell 6.")
else:
    print("═" * 62)
    print("COVERAGE")
    print("═" * 62)
    print(f"  Signals            {len(pen):,}")
    print(f"  Tickers            {pen['symbol'].nunique()}")
    print(f"  Sessions           {pen['session'].nunique()}")
    print(f"  Date span          {pen['session'].min()} → {pen['session'].max()}")

    if 'in_playbook' in pen and pen['in_playbook'].nunique() > 1:
        print()
        print("  Playbook names vs broad universe — do NOT pool these:")
        print(pen.groupby('in_playbook').agg(
            signals=('reached_vwap','size'),
            tickers=('symbol','nunique'),
            hit_vwap=('reached_vwap','mean')).round(3).to_string())
    print()
    print(pen.groupby(['tf','path','direction']).size()
            .rename('signals').to_frame())

    rec = pen[pen['reached_vwap']]
    fail = pen[~pen['reached_vwap']]

    print()
    print("═" * 62)
    print("OUTCOME SPLIT")
    print("═" * 62)
    print(f"  Reached VWAP       {len(rec):,}  ({len(rec)/len(pen)*100:.1f}%)")
    print(f"  Never reached      {len(fail):,}  ({len(fail)/len(pen)*100:.1f}%)")
    print(f"  Gapped past stop   {pen['gapped'].sum():,}  "
          f"({pen['gapped'].mean()*100:.1f}%)  ← brake can't fill here")

    qs = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

    print()
    print("═" * 62)
    print("PENETRATION BEFORE VWAP — trades that WORKED")
    print("as a fraction of the signal candle's own range")
    print("═" * 62)
    for tf_label in ("2m", "5m"):
        sub = rec[rec['tf'] == tf_label]['pen_before_vwap_norm'].dropna()
        if len(sub) == 0:
            continue
        print(f"\n  {tf_label}  (n={len(sub):,})")
        print(f"    never poked at all : {(sub == 0).mean()*100:5.1f}%")
        for q in qs:
            print(f"    {int(q*100):>3}th pct          : {sub.quantile(q)*100:6.1f}% of range")

    print()
    print("═" * 62)
    print("PENETRATION — trades that FAILED (never hit VWAP)")
    print("═" * 62)
    for tf_label in ("2m", "5m"):
        sub = fail[fail['tf'] == tf_label]['pen_session_norm'].dropna()
        if len(sub) == 0:
            continue
        print(f"\n  {tf_label}  (n={len(sub):,})")
        for q in [0.25, 0.5, 0.75]:
            print(f"    {int(q*100):>3}th pct          : {sub.quantile(q)*100:6.1f}% of range")

    print()
    print("═" * 62)
    print("BRAKE TRADEOFF — what each depth costs and saves")
    print("═" * 62)
    print(f"  {'brake':>7}  {'winners kept':>13}  {'losers cut early':>17}")
    grid = []
    for d in [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50,0.75,1.00]:
        kept = (rec['pen_before_vwap_norm'] <= d).mean()
        cut  = (fail['pen_session_norm']    >= d).mean()
        grid.append({'brake_pct_of_range': d,
                     'winners_kept': kept,
                     'losers_cut_early': cut})
        print(f"  {d*100:>6.0f}%  {kept*100:>12.1f}%  {cut*100:>16.1f}%")

    print()
    print("  Read it as: a brake that keeps most winners while still")
    print("  cutting most losers is the knee. If no depth does both,")
    print("  the two distributions overlap and the brake is a pure")
    print("  cost dial rather than a discriminator — worth knowing.")

    pd.DataFrame(grid).to_csv(SUMMARY_OUT, index=False)
    print(f"\n✅ Saved → {SUMMARY_OUT}")